# 05. 리랭커 성능 측정

## 무엇을 재는 건가

검색은 지금 이렇게 돈다.

```
공고 2,153건 ──[빠른 검색]──▶ 상위 30건 ──▶ 상위 3건을 화면에
```

여기에 리랭커를 끼우면 이렇게 된다.

```
공고 2,153건 ──[빠른 검색]──▶ 30건 ──[리랭커가 다시 읽고 순서 바꿈]──▶ 상위 3건
```

**뒤쪽에 밀려 있던 좋은 공고를 위로 끌어올리는 것**이 목적이다.
그게 실제로 되는지 숫자로 확인한다.

## 무엇끼리 비교하나

| 이름 | 설명 |
| --- | --- |
| `dense` | 지금 서비스가 쓰는 방식 (의미 검색만) |
| `rrf` | 의미 검색 + 단어 검색을 섞은 것 |
| `rrf+rerank` (사전학습) | 남이 만든 리랭커를 그대로 얹은 것 |
| `rrf+rerank` (학습) | **우리가 가르친 리랭커**를 얹은 것 |

세 번째와 네 번째를 비교해야 **"가르친 게 효과가 있었나"** 를 알 수 있다.

## 시험 문제는 20개만 쓴다

60개 중 40개는 리랭커를 가르칠 때 썼다. 그걸로 시험 보면 의미가 없으므로
**한 번도 안 보여준 20개로만** 잰다.

---

## 실행 전 준비

1. `04_reranker_train.py` 를 먼저 돌려 모델을 만든다
2. 벡터 색인이 이 PC 에 있어야 한다 (아래 칸이 확인·생성한다)

In [1]:
# -*- coding: utf-8 -*-
import io, os, sys, json, time, collections

ROOT = os.path.abspath('..')
if os.path.basename(os.getcwd()) == 'ml':
    os.chdir(ROOT)

# 모델을 아직 안 받았다면 내려받아야 하므로 오프라인 모드를 끈다.
# (vecstore.py 가 HF_HUB_OFFLINE 을 1 로 기본 설정하는데, 미리 0 을 넣어 막는다)
os.environ.setdefault('HF_HUB_OFFLINE', '0')

sys.path[:0] = [ROOT, os.path.join(ROOT, 'eval'), os.path.join(ROOT, 'ml')]

import rerank_common as rc
print('작업 폴더 :', os.getcwd())
print('학습한 모델 :', '있음' if os.path.exists(rc.ADAPTER_DIR) else '없음 ← 04 를 먼저 돌린다')

작업 폴더 : C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection
학습한 모델 : 있음


### 벡터 색인 확인

리랭커에 넣을 후보 30건을 뽑으려면 기존 검색이 돌아야 한다.
검색은 벡터 색인(Chroma)을 쓰는데, 이 파일은 git 에 올리지 않으므로
**이 PC 에서 한 번 만들어야** 한다. EC2 DB 의 벡터를 그대로 옮기는 것이라
몇 초면 끝난다.

In [19]:
import chromadb

store = os.path.join(ROOT, 'data', 'vecstore', 'chroma')
need = True
if os.path.exists(store):
    try:
        col = chromadb.PersistentClient(path=store).get_collection('notices_v1')
        print('색인 있음 · %d건' % col.count())
        need = col.count() < 100
    except Exception as e:
        print('색인을 열지 못했다:', e)

if need:
    print('색인을 만든다 (EC2 DB 의 벡터를 옮긴다)...')
    import ec2_vecstore
    ec2_vecstore.run(rebuild=True)
    print('완료')

색인 있음 · 2154건


---

## 1. 시험 문제 20개 준비

In [20]:
import common
import evaluate as ev

splits = rc.load_splits()
all_queries = common.load_queries()
test_queries = {qid: all_queries[qid] for qid in splits['test'] if qid in all_queries}

qrels = rc.load_qrels()

kinds = collections.Counter(q['kind'] for q in test_queries.values())
print('시험 질의 %d개 (정상 %d · 무관 %d)'
      % (len(test_queries), kinds['normal'], kinds['negative']))
print('학습에 쓴 질의 %d개는 여기에 없다.' % len(splits['train']))

시험 질의 20개 (정상 17 · 무관 3)
학습에 쓴 질의 40개는 여기에 없다.


### 공고 문장을 미리 다 받아둔다

리랭커는 후보 30건의 **본문**을 읽어야 한다. 이걸 질의가 올 때마다 DB 에서
가져오면 접속을 수십 번 하게 되고, 원격 DB 라 중간에 끊긴다.

그래서 **여기서 한 번에 다 받아 메모리에 올려둔다.** 2,153건이라 몇 MB 뿐이다.
이 뒤로는 DB 를 건드리지 않는다.

In [21]:
t0 = time.time()
all_notices = common.load_notices(None, with_attachment=False)
NOTICE_TEXTS = {nid: rc.notice_text(n) for nid, n in all_notices.items()}
del all_notices

sample = next(iter(NOTICE_TEXTS.values()))
print('공고 %d건 · %.1f초' % (len(NOTICE_TEXTS), time.time() - t0))
print('평균 길이 %d자' % (sum(len(t) for t in NOTICE_TEXTS.values()) // len(NOTICE_TEXTS)))
print()
print('리랭커가 읽는 문장은 이렇게 생겼다:')
print(' ', sample[:200], '...')

공고 2153건 · 0.3초
평균 길이 328자

리랭커가 읽는 문장은 이렇게 생겼다:
  영세개인사업자의 체납액 징수특례 제도 안내 기타 제도 소상공인 영세 개인사업자가 폐업 후 다시 사업을 시작하거나 취업한 경우 무재산 등 사유로 징수가 곤란한 체납액에 대하여 가산금ㆍ납부지연가산세 면제 및 최대 5년까지 분납을 허용하는 제도입니다. ☞ 22.12.31. 까지 모든 개인사업 폐업 및 사업재기 및 취업 후 3개월 이상 근무중인 영세 개인사업자 ☞ ...


---

## 2. 리랭커를 기존 평가 코드에 끼우기

**지표를 새로 만들지 않는다.** `eval/evaluate.py` 가 이미 Recall@3 · nDCG@3 를
계산하고 있고, 기존 성적(dense 0.494 / rrf 0.577)도 그 코드로 나온 값이다.
여기서 다시 구현하면 숫자를 비교할 수 없게 된다.

그래서 기존 `Systems` 를 물려받아 **`+rerank` 라는 이름만 추가**한다.
나머지 동작은 손대지 않는다.

In [22]:
class RerankSystems(ev.Systems):
    """기존 검색에 리랭커 한 단계를 덧붙인다.

    'rrf+rerank' 처럼 이름 뒤에 +rerank 를 붙이면,
    원래 방식으로 후보 POOL 건을 뽑은 뒤 리랭커가 순서를 다시 매긴다.
    """

    POOL = 30      # 리랭커가 다시 읽을 후보 수

    def __init__(self, scorer, texts):
        super().__init__()
        self.scorer = scorer
        self.texts = texts        # 미리 받아둔 {공고id: 문장}. DB 를 다시 안 본다
        self.elapsed = []

    def run(self, name, q, k=30):
        if not name.endswith('+rerank'):
            return super().run(name, q, k)
        base = name[:-len('+rerank')]
        hits = super().run(base, q, k=self.POOL)
        ids = [n for n, _ in hits]
        if not ids:
            return []
        t0 = time.time()
        texts = [self.texts.get(i, '') for i in ids]
        scores = self.scorer.score(common.query_text(q), texts)
        self.elapsed.append((time.time() - t0) * 1000)
        return sorted(zip(ids, scores), key=lambda x: -x[1])[:k]

print('준비됨. 후보 %d건을 다시 읽는다.' % RerankSystems.POOL)

준비됨. 후보 30건을 다시 읽는다.


---

## 3. 기준선 — 리랭커 없이

먼저 지금 방식의 성적을 시험 20문제로 잰다. 이게 비교 대상이다.

**지표 읽는 법**

| 이름 | 뜻 |
| --- | --- |
| `P@3(2)` | 화면에 보이는 3건 중 "딱 맞음" 비율. **주 지표** |
| `nDCG@3` | 좋은 것이 위쪽에 있을수록 높다 |
| `미판정@3` | 3건 중 정답지에 없는 것의 비율. 높으면 점수를 못 믿는다 |

In [23]:
results = []

def measure(label, system_name, systems):
    per = ev.evaluate(system_name, test_queries, qrels, systems)
    s = ev.summarize(label, per)
    s['_per'] = per
    results.append(s)
    print('%-26s P@3(≥1) %.3f · P@3(2) %.3f · nDCG@3 %.3f · 미판정 %.3f'
          % (label, s['p3_rel'], s['p3_rec'], s['ndcg3'], s['unjudged3']))
    return s

plain = ev.Systems()
measure('dense (현재 서비스)', 'dense', plain)
measure('rrf (의미+단어)', 'rrf', plain)

dense (현재 서비스)             P@3(≥1) 0.706 · P@3(2) 0.549 · nDCG@3 0.695 · 미판정 0.098
rrf (의미+단어)                P@3(≥1) 0.725 · P@3(2) 0.627 · nDCG@3 0.718 · 미판정 0.098


{'system': 'rrf (의미+단어)',
 'queries': 17,
 'p3_rec': 0.6274509803921569,
 'p3_rec_judged': 0.6764705882352942,
 'p3_rel': 0.7254901960784315,
 'ndcg3': 0.718204911966907,
 'unjudged3': 0.0980392156862745,
 'p3_rec_ci95': (0.4509803921568627, 0.7843137254901961),
 'distinct_top3': 46,
 'max_repeat': ('kstartup:178831', 3),
 'group_slots': 3,
 'negative_group_slots': 2,
 '_per': [{'qid': 'n002',
   'kind': 'negative',
   'category': '무관',
   'top1_score': 0.030309988518943745,
   'top3': [{'notice_id': 'bizinfo:PBLN_000000000126372', 'topic_rel': 0},
    {'notice_id': 'kstartup:178453', 'topic_rel': None},
    {'notice_id': 'kstartup:178786', 'topic_rel': 0}],
   'p3_rec': 0.0,
   'p3_rel': 0.0,
   'p3_rec_judged': 0.0,
   'ndcg3': 0.0,
   'unjudged3': 0.3333333333333333,
   'group3': 0},
  {'qid': 'n007',
   'kind': 'negative',
   'category': '무관',
   'top1_score': 0.029116045245077504,
   'top3': [{'notice_id': 'kstartup:179095', 'topic_rel': 0},
    {'notice_id': 'bizinfo:PBLN_0000000

---

## 4. 사전학습 리랭커 — 가르치기 전

남이 만들어 둔 리랭커를 그대로 얹는다. **우리 데이터로 가르치지 않은 상태**다.

처음 실행하면 모델을 내려받는다(약 2.2GB). 한 번만 받으면 다음부터는 빠르다.

In [24]:
t0 = time.time()
base_scorer = rc.Scorer()                   # adapter 없음 = 사전학습 그대로
print('모델 준비 %.0f초 · 장치 %s' % (time.time() - t0, base_scorer.device))

base_sys = RerankSystems(base_scorer, NOTICE_TEXTS)
measure('rrf+rerank (사전학습)', 'rrf+rerank', base_sys)
print('질의당 리랭킹 시간 : 평균 %.0f ms' % (sum(base_sys.elapsed) / len(base_sys.elapsed)))

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 736.55it/s]


모델 준비 4초 · 장치 cuda
rrf+rerank (사전학습)          P@3(≥1) 0.784 · P@3(2) 0.608 · nDCG@3 0.780 · 미판정 0.118
질의당 리랭킹 시간 : 평균 1076 ms


---

## 5. 우리가 가르친 리랭커

`04_reranker_train.py` 가 만든 모델을 얹는다. 위와 **후보도 같고 조건도 같다.**
다른 건 리랭커가 우리 데이터를 배웠다는 것뿐이다.

In [ ]:
tuned_scorer = rc.Scorer(adapter=rc.ADAPTER_DIR)
tuned_sys = RerankSystems(tuned_scorer, NOTICE_TEXTS)
measure('rrf+rerank (학습)', 'rrf+rerank', tuned_sys)
print('질의당 리랭킹 시간 : 평균 %.0f ms' % (sum(tuned_sys.elapsed) / len(tuned_sys.elapsed)))

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 714.44it/s]


---

## 6. 결과 한눈에

In [ ]:
print('%-26s %-10s %-10s %-10s %-10s %-9s'
      % ('시스템', 'P@3(≥1)', 'P@3(2)', '판정칸만', 'nDCG@3', '미판정@3'))
print('-' * 86)
for s in results:
    judged = s['p3_rec_judged']
    print('%-26s %-10.3f %-10.3f %-10s %-10.3f %-9.3f'
          % (s['system'], s['p3_rel'], s['p3_rec'],
             '%.3f' % judged if judged is not None else '  -  ',
             s['ndcg3'], s['unjudged3']))

print()
print('P@3(≥1)  주 지표. 「관련 있음」 기준 — 사람·LLM 판정 일치율 0.80 구간이다.')
print('P@3(2)   참고. 「딱 맞음」 기준 — 일치율 0.64 로 기준 미달이라 단독으로 읽지 않는다.')
print('미판정@3 0.10 을 넘으면 그 줄의 숫자를 믿지 않는다 (evaluate.py 원칙).')

### 차이가 진짜인지 확인

질의가 20개뿐이라 **점수가 조금 올라간 것은 우연일 수 있다.**
그래서 질의별로 짝지어 비교하고 95% 구간을 본다.

**구간이 0 을 걸치면 "좋아졌다고 말할 수 없다"** 는 뜻이다.
이걸 확인하지 않고 "올랐습니다"라고 쓰면 안 된다.

In [ ]:
base = results[1]      # rrf 를 기준으로 삼는다

for metric, title in (('p3_rel', 'P@3(≥1) — 주 지표'), ('p3_rec', 'P@3(2) — 참고')):
    print('== %s ==   기준 : %s' % (title, base['system']))
    for s in results:
        if s is base:
            continue
        d = ev.paired_diff([r[metric] for r in base['_per'] if r['kind'] == 'normal'],
                           [r[metric] for r in s['_per'] if r['kind'] == 'normal'])
        if not d:
            continue
        mid, lo, hi = d
        verdict = ('개선이라 볼 수 있다' if lo > 0 else
                   '나빠졌다' if hi < 0 else '차이를 단정할 수 없다')
        print('  %-26s %+.3f  (%+.3f ~ %+.3f)   → %s' % (s['system'], mid, lo, hi, verdict))
    print()

---

## 7. 학습 과정 그림

손실(loss)은 **틀린 정도**다. 낮을수록 좋다.

- 학습 손실만 내려가고 시험 손실이 올라가면 → **외워버린 것**(과적합)
- 둘 다 내려가면 → 제대로 배운 것

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt

info = json.load(io.open(os.path.join(rc.ADAPTER_DIR, 'train_info.json'), encoding='utf-8'))
h = info['history']

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot([r['epoch'] for r in h], [r['train_loss'] for r in h], marker='o', label='학습 손실')
ax.plot([r['epoch'] for r in h], [r['test_loss'] for r in h], marker='s', label='시험 손실')
ax.set_xlabel('바퀴(epoch)'); ax.set_ylabel('손실'); ax.legend(); ax.grid(alpha=.3)
ax.set_title('리랭커 학습 곡선')
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'ml', 'data', 'reranker_loss.png'), dpi=150)
plt.show()

print('학습 설정 :', {k: info[k] for k in ('epochs', 'effective_batch', 'lr', 'lora_rank')})
print('학습한 부품 : %s개 / 전체 %s개 (%.2f%%)'
      % (f"{info['trainable_params']:,}", f"{info['total_params']:,}",
         info['trainable_params'] / info['total_params'] * 100))

---

## 8. 문서에 넣을 숫자

「학습한 ML/DL 모델」 문서의 **3.1 저장·추론** 표에 그대로 들어갈 값들이다.

In [ ]:
def dir_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

adapter_mb = dir_size(rc.ADAPTER_DIR) / 1024**2
clf = os.path.join(ROOT, 'ml', 'models', 'age_classifier_v1.joblib')

rows = [
    ('업력 분류기 크기', '%.1f' % (os.path.getsize(clf) / 1024) if os.path.exists(clf) else '-', 'KB'),
    ('리랭커 LoRA 크기', '%.1f' % adapter_mb, 'MB'),
    ('리랭커 후보 수', str(RerankSystems.POOL), '건'),
    ('리랭킹 시간 (GPU)', '%.0f' % (sum(tuned_sys.elapsed) / len(tuned_sys.elapsed)), 'ms/질의'),
]
for name, value, unit in rows:
    print('%-22s %10s %s' % (name, value, unit))

if tuned_scorer.device == 'cuda':
    import torch
    print('%-22s %10.1f GB' % ('GPU 최대 사용량', torch.cuda.max_memory_allocated() / 1024**3))

---

## 9. 결과 저장

문서를 쓸 때 다시 돌리지 않아도 되도록 파일로 남긴다.

In [ ]:
out = os.path.join(ROOT, 'ml', 'data', 'reranker_results.json')
json.dump({
    'test_queries': len(test_queries),
    'pool': RerankSystems.POOL,
    'train_info': info,
    'systems': [{k: v for k, v in s.items() if k != '_per'} for s in results],
}, io.open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
print('저장 →', out)

---

## 정리

만들어진 것

- `ml/models/reranker_lora/` — 학습한 딥러닝 모델 (제출물)
- `ml/data/reranker_results.json` — 성능 비교 결과
- `ml/data/reranker_loss.png` — 학습 곡선 그림

**결과를 어떻게 읽든 결과서는 쓸 수 있다.**

| 나온 결과 | 결과서에 쓸 내용 |
| --- | --- |
| 학습한 쪽이 더 좋다 | 효과를 확인했다. 서비스 적용을 검토한다 |
| 사전학습과 비슷하다 | 893쌍으로는 부족했다. 필요한 데이터 규모를 가늠했다 |
| 오히려 나빠졌다 | 과적합. 학습 곡선으로 원인을 보인다 |

실험은 좋게 나와야만 가치가 있는 게 아니다. **왜 그런지 설명할 수 있으면 된다.**